In [11]:
import ast
import os
import json
import requests

import pandas as pd
import dotenv
import redis
import numpy as np

from functools import reduce

In [12]:
# Convert data from byte into datatpyes
def convert_from_byte(byte_dict):
    return {key.decode('utf-8'): value.decode('utf-8') for key, value in byte_dict.items()}

In [13]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [14]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

## VITC

In [15]:
# Only do this once
# Import VITC daraset
with open("vitc_evaluation_sup_ref.json") as f:
    vitc = json.load(f)

# Randomise order
np.random.shuffle(vitc)
# Test labels
labels = []
for claim in vitc:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array(['REFUTES', 'SUPPORTS'], dtype='<U8'), array([217, 283]))

In [16]:
# Only do this once
# Populate Vercel KV with vitc datasets
for datapoint in vitc:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': datapoint['label']
    })

KeyboardInterrupt: 

In [ ]:
# Only do this once
vitc_ids = [datapoint['claim_id'] for datapoint in vitc]

In [ ]:
# Only do this once
# Save ids as json 
with open('vitc_ids.json', 'w') as f:
    json.dump(vitc_ids, f)

In [7]:
# OK after randomising once at so on, use vitc ids saved in json
with open('vitc_ids.json') as f:
    vitc_ids = json.load(f)

In [8]:
# Create three batches for VITC
# 100 samples are included in all batches to test for inter-rater reliability

repeated_samples = vitc_ids[:100]

vitc_batches = {
    'vitc_repeated': repeated_samples 
}
start_index = 100
for i in range(3):
    end_index = start_index + ((len(vitc_ids) - 100) // 3) if i < 2 else len(vitc_ids)
    print(f'{start_index} - {end_index}')
    unique_samples = vitc_ids[start_index:end_index]
    start_index = end_index
    vitc_batches[f'vitc{i+1}_workpackage1'] = unique_samples[:15]
    vitc_batches[f'vitc{i+1}_workpackage3'] = unique_samples[15:]

for key in vitc_batches:
    print(key, len(vitc_batches[key])) 

100 - 233
233 - 366
366 - 500
vitc_repeated 100
vitc1_workpackage1 15
vitc1_workpackage3 118
vitc2_workpackage1 15
vitc2_workpackage3 118
vitc3_workpackage1 15
vitc3_workpackage3 119


In [16]:
# Populate vercel KV with VITC batches
for batch_id in vitc_batches.keys():
    claim_ids = vitc_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [ ]:
# Assign batches to annotators
vitc_annotators = {
    'test': 'vitc1',
    'mahmud': 'vitc1',
    'sara': 'vitc1',
    'chris': 'vitc2',
    'vishal': 'vitc3',
}

In [87]:
# Upload annotators to Vercel KV
for annotator_id in vitc_annotators.keys():
    r.hset(annotator_id, mapping={
        'stage': 'workpackage1',
        'stage_to_batch': json.dumps({
            'workpackage1': f'{vitc_annotators[annotator_id]}_workpackage1',
            'workpackage2': 'vitc_repeated',
            'workpackage3': f'{vitc_annotators[annotator_id]}_workpackage3'
        }),
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

In [22]:
r.hset('mahmud', mapping={'workpackage1': 'vitc1', 'stage': 'workpackage1', 'workpackage1_progress': 0, 'workpackage2_progress': 0, 'workpackage3_progress': 0})

0

In [24]:
# Replace vitc_15319 with new datapoint, but keep id the same
replacement_data = {
    "claim": "Fibromyalgia can make it hard to get out of bed.",
    "evidence": "Fibromyalgia is a medical condition defined by the presence of chronic widespread pain, fatigue, waking unrefreshed, cognitive symptoms, lower abdominal pain or cramps, and depression. Other symptoms include insomnia and a general hypersensitivity. The cause of fibromyalgia is unknown, but is believed to involve a combination of genetic and environmental factors.",
    "label": "SUPPORTS"
}

r.hset('vitc_15319', mapping=replacement_data)

0

In [28]:
# get batch that sara is assigned to
sara_batch = r.hget('sara', 'stage_to_batch')
sara_batch = json.loads(sara_batch)
sara_wp3 = sara_batch['workpackage3']

In [ ]:
# get ids of claims in sara's batch
sara_batch_ids = r.hget(sara_wp3, 'claim_ids')
sara_batch_ids = json.loads(sara_batch_ids)

In [36]:
# Send Sara's wp3 to mahmoud because she likely won't be able to finish it 
vitc1_wp3 = [claim for claim in vitc if claim['claim_id'] in sara_batch_ids]
vitc1_wp3_df = pd.DataFrame(vitc1_wp3)

In [ ]:
# export to csv
vitc1_wp3_df.to_csv('vitc1_wp3_(Sara).csv', index=True)

,claim_id,claim,evidence,label,sim_score,bert_f1,bleurt
0,vitc_460,Rob Huebel stars in The House .,"The film stars Will Ferrell , Amy Poehler , Ja...",SUPPORTS,0.429347,-0.069176,0.355481
1,vitc_503,The House grossed more than $ 31 million world...,"The film was released on June 30 , 2017 , by W...",SUPPORTS,0.518636,0.230056,0.412786
2,vitc_2659,Blazing Saddles was reviewed by fewer than 50 ...,On the film-critics aggregator Rotten Tomatoes...,SUPPORTS,0.429546,0.176192,0.529010
3,vitc_3264,The relationship between Lyft and drivers has ...,Lyft then retains approximately 20-25 % ( 20 %...,SUPPORTS,0.644001,0.058576,0.257463
4,vitc_1899,Tragic Kingdom was the last album in which Eri...,It is the last album to feature keyboardist Er...,SUPPORTS,0.674240,0.236486,0.259031
...,...,...,...,...,...,...,...
113,vitc_9180,A & E is an American television channel .,A & E -LRB- previously Arts & Entertainment Ne...,SUPPORTS,0.560509,0.135991,0.391061
114,vitc_1418,The song All I Want for Christmas Is You reach...,"The song was commercially successful , topping...",REFUTES,0.435714,-0.046520,0.285371
115,vitc_1861,Black Mountain has less than three LPs .,"Mountain has released three LPs , Black Mounta...",REFUTES,0.490144,-0.033957,0.293262
116,vitc_12729,Sarawak 's city with the most people is Kuching .,"Kuching -LSB- ˈkuːtʃɪŋ -RSB- -LRB- Jawi : , -R...",SUPPORTS,0.827684,-0.006636,0.286391


## Phemplus

In [5]:
phemeplus_annotators = {
    'bleiz': 'phemeplus1',
    'nelly': 'phemeplus2',
    'yazhou': 'phemeplus3'
}

In [8]:


# Import phemeplus dataset
with open("phemeplus_incomplete_15-11-24.json") as f:
    phemeplus = json.load(f)

# Randomise order
np.random.shuffle(phemeplus)
# Test labels
labels = []
for claim in phemeplus:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array([False,  True]), array([ 91, 202]))

In [124]:
# Populate Vercel KV with phemeplus datasets
for datapoint in phemeplus:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': "true" if datapoint['label'] else "false"
    })

In [145]:
# Create three batches for PHEMEPLUS workpackage 1 (phemeplus is incomplete so far)
repeated_samples = phemeplus_ids[:100]
workpackage1_samples1 = phemeplus_ids[100:115]
workpackage1_samples2 = phemeplus_ids[115:130]
workpackage1_samples3 = phemeplus_ids[130:145]

phemeplus_batches = {
    'phemeplus_repeated': repeated_samples,
    'phemeplus1_workpackage1': workpackage1_samples1,
    'phemeplus2_workpackage1': workpackage1_samples2,
    'phemeplus3_workpackage1': workpackage1_samples3
}

for key in phemeplus_batches:
    print(key, len(phemeplus_batches[key]))


phemeplus_repeated 100
phemeplus1_workpackage1 15
phemeplus2_workpackage1 15
phemeplus3_workpackage1 15


In [147]:
# Populate vercel KV with phemeplus batches
for batch_id in phemeplus_batches.keys():
    claim_ids = phemeplus_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [151]:
# Upload annotators to Vercel KV
for annotator_id in phemeplus_annotators.keys():
    r.hset(annotator_id, mapping={
        'stage': 'workpackage1',
        'stage_to_batch': json.dumps({
            'workpackage1': f'{phemeplus_annotators[annotator_id]}_workpackage1',
            'workpackage2': 'phemeplus_repeated',
            # 'workpackage3': f'{vitc_annotators[annotator_id]}_workpackage3'
        }),
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

### Add workpackage 3 for which dataset was provided at later point

In [9]:
# check batch ids that have been completed
# import workpackage1 datasets
with open('bleiz_workpackage1.json') as f:
    bleiz_workpackage1 = json.load(f)
with open('nelly_workpackage1.json') as f:
    nelly_workpackage1 = json.load(f)
with open('yazhou_workpackage1.json') as f:
    yazhou_workpackage1 = json.load(f)

# import workpackage2 datasets
with open('bleiz_workpackage2.json') as f:
    bleiz_workpackage2 = json.load(f)
with open('nelly_workpackage2.json') as f:
    nelly_workpackage2 = json.load(f)
with open('yazhou_workpackage2.json') as f:
    yazhou_workpackage2 = json.load(f)


In [17]:
# Extract claim_ids from completed workpackages
bleiz_workpackage1_ids = [datapoint['claim_id'] for datapoint in bleiz_workpackage1]
nelly_workpackage1_ids = [datapoint['claim_id'] for datapoint in nelly_workpackage1]
yazhou_workpackage1_ids = [datapoint['claim_id'] for datapoint in yazhou_workpackage1]
bleiz_workpackage2_ids = [datapoint['claim_id'] for datapoint in bleiz_workpackage2]
nelly_workpackage2_ids = [datapoint['claim_id'] for datapoint in nelly_workpackage2]
yazhou_workpackage2_ids = [datapoint['claim_id'] for datapoint in yazhou_workpackage2]

# Concatenate id lists
completed_workpackage_ids = reduce(lambda x, y: x + y, [bleiz_workpackage1_ids, 
                                                        nelly_workpackage1_ids, 
                                                        yazhou_workpackage1_ids, 
                                                        bleiz_workpackage2_ids, 
                                                        nelly_workpackage2_ids, 
                                                        yazhou_workpackage2_ids])

# Only get unique ids
completed_workpackage_ids = np.unique(completed_workpackage_ids)

In [18]:
# open original phemeplus dataset
with open("phemeplus_incomplete_15-11-24.json") as f:
    phemeplus = json.load(f)

In [20]:
# extract claims that have not been completed
phemeplus_incomplete = [datapoint for datapoint in phemeplus if datapoint['claim_id'] not in completed_workpackage_ids]

In [22]:
# add extra phemeplus datapoints for work package 3
with open("phemeplus_rest.json") as f:
    phemeplus_rest = json.load(f)

In [27]:
# concatenate phemeplus_incomplete with phemeplus_rest
phemeplus_wp3 = phemeplus_incomplete + phemeplus_rest

# Randomise order
np.random.shuffle(phemeplus_wp3)
# Test labels
labels = []
for claim in phemeplus_wp3:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array([False,  True]), array([108, 247]))

In [28]:
# only do this one
# save order of phemeplus_wp3
phemeplus_wp3_ids = [datapoint['claim_id'] for datapoint in phemeplus_wp3]
with open('phemeplus_wp3_ids.json', 'w') as f:
    json.dump(phemeplus_wp3_ids, f)

In [29]:
# load ids from json
with open('phemeplus_wp3_ids.json') as f:
    phemeplus_wp3_ids = json.load(f)

In [33]:
# divide phemeplus_wp3 into batches
interval = len(phemeplus_wp3_ids)//3

phemeplus1_workpackage3 = phemeplus_wp3_ids[:interval]
phemeplus2_workpackage3 = phemeplus_wp3_ids[interval:2*interval]
phemeplus3_workpackage3 = phemeplus_wp3_ids[2*interval:]

In [53]:
def calculate_batch_character_length(batch, dataset):
    character_lengths = []
    for claim_id in batch:
        for datapoint in dataset:
            if datapoint['claim_id'] == claim_id:
                character_lengths.append(len(datapoint['claim']) + len(datapoint['evidence']))
                break
    return sum(character_lengths)

In [55]:
print(calculate_batch_character_length(phemeplus1_workpackage3, phemeplus_wp3))
print(calculate_batch_character_length(phemeplus2_workpackage3, phemeplus_wp3))
print(calculate_batch_character_length(phemeplus3_workpackage3, phemeplus_wp3))

199715
175916
188045


In [58]:
# ok Mahmoud wants pehemplus3_workpackage3 assigned to Nelly so workpackage 2 and 3 have to swapped
temp = phemeplus3_workpackage3

phemeplus3_workpackage3 = phemeplus2_workpackage3
phemeplus2_workpackage3 = temp

In [56]:
# add remaining claims to Vercel KV
for datapoint in phemeplus_wp3:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': "true" if datapoint['label'] else "false"
    })

In [59]:
# add batches to Vercel KV
r.hset('phemeplus1_workpackage3', mapping={'claim_ids': json.dumps(phemeplus1_workpackage3)})
r.hset('phemeplus2_workpackage3', mapping={'claim_ids': json.dumps(phemeplus2_workpackage3)})
r.hset('phemeplus3_workpackage3', mapping={'claim_ids': json.dumps(phemeplus3_workpackage3)})


0

In [62]:
# update stage_to_batch for annotators

for i,name in enumerate(['bleiz', 'nelly', 'yazhou']):

    r.hset(name, mapping={'stage_to_batch': json.dumps({
        'workpackage1': f'phemeplus{i+1}_workpackage1',
        'workpackage2': f'phemeplus{i+1}_workpackage2',
        'workpackage3': f'phemeplus{i+1}_workpackage3'
    })})

In [63]:
# check submission 
test_data = r.hgetall('nelly')
test_data = convert_from_byte(test_data)
test_data['stage_to_batch']

'{"workpackage1": "phemeplus2_workpackage1", "workpackage2": "phemeplus2_workpackage2", "workpackage3": "phemeplus2_workpackage3"}'

## Climate Fever

In [6]:
# Import VITC daraset
with open("clfever.json") as f:
    clfever = json.load(f)

# Randomise order
np.random.shuffle(clfever)
# Test labels
labels = []
for claim in clfever:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array(['REFUTES', 'SUPPORTS'], dtype='<U8'), array([143, 357]))

In [ ]:
# Only do this once
# Populate Vercel KV with vitc datasets
for datapoint in clfever:
    id = datapoint['claim_id']
    r.hset(id, mapping={
        'claim': datapoint['claim'],
        'evidence': datapoint['evidence'],
        'label': datapoint['label']
    })

In [7]:
# Only do this once
clfever_ids = [datapoint['claim_id'] for datapoint in clfever]

In [8]:
# Only do this once
# Save ids as json 
with open('clfever_ids.json', 'w') as f:
    json.dump(clfever_ids, f)

In [9]:
# OK after randomising once at so on, use vitc ids saved in json
with open('clfever_ids.json') as f:
    clfever_ids = json.load(f)

In [12]:
# Create three batches for Climate Fever
# 100 samples are included in all batches to test for inter-rater reliability

repeated_samples = clfever_ids[:100]

clfever_batches = {
    'clfever_repeated': repeated_samples 
}
start_index = 100
for i in range(3):
    end_index = start_index + ((len(clfever_ids) - 100) // 3) if i < 2 else len(clfever_ids)
    print(f'{start_index} - {end_index}')
    unique_samples = clfever_ids[start_index:end_index]
    start_index = end_index
    clfever_batches[f'clfever{i+1}_workpackage1'] = unique_samples[:15]
    clfever_batches[f'clfever{i+1}_workpackage3'] = unique_samples[15:]

for key in clfever_batches:
    print(key, len(clfever_batches[key])) 

100 - 233
233 - 366
366 - 500
clfever_repeated 100
clfever1_workpackage1 15
clfever1_workpackage3 118
clfever2_workpackage1 15
clfever2_workpackage3 118
clfever3_workpackage1 15
clfever3_workpackage3 119


In [13]:
# Populate vercel KV with Climate Fever batches
for batch_id in clfever_batches.keys():
    claim_ids = clfever_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

In [18]:
# Assign batches to annotators
clfever_annotators = {
    'clfever_test': 'clfever1',
    'yuli': 'clfever1',
    'jorge': 'clfever2',
    'anel': 'clfever3',    
}

In [19]:
# Upload annotators to Vercel KV
for annotator_id in clfever_annotators.keys():
    r.hset(annotator_id, mapping={
        'stage': 'workpackage1',
        'stage_to_batch': json.dumps({
            'workpackage1': f'{clfever_annotators[annotator_id]}_workpackage1',
            'workpackage2': 'clfever_repeated',
            'workpackage3': f'{clfever_annotators[annotator_id]}_workpackage3'
        }),
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

In [21]:
r.hset('clfever_test', mapping={'stage': 'workpackage1', 'workpackage1_progress': 0, 'workpackage2_progress': 0, 'workpackage3_progress': 0})

0

In [5]:
# reset yuli
r.hset('yuli', mapping={'stage': 'workpackage3', 'workpackage3_progress': 0})

0

## Rerun 

Climate Fever repeated batch has to be rerun twice 

In [10]:
# Bleiz and Chris did the rerun
# Now it is the turn of Guneet, Cian and Iqra (also reset Mahmud)

rerun_ids = ['guneet', 'cian', 'iqra', 'mahmud']

for id in rerun_ids:
    r.hset(id, mapping={
        'stage_to_batch': json.dumps({
            'workpackage2': 'clfever_repeated',
        }),
        'stage': 'workpackage2',
        'workpackage1_progress': 0,
        'workpackage2_progress': 0,
        'workpackage3_progress': 0
    })

## save data

In [19]:
# check submission 
test_data = r.hgetall('iqra')
test_data = convert_from_byte(test_data)
test_data['workpackage2_progress']

'100'

In [20]:
def save_data(id, workpackage_id):
    data = r.hgetall(id)
    data = convert_from_byte(data)
    # Get ids for workpackages
    workpackage_mapping = ast.literal_eval(data['stage_to_batch'])
    workpackage = workpackage_mapping[workpackage_id]
    claim_ids = ast.literal_eval(convert_from_byte(r.hgetall(workpackage))['claim_ids'])
    dataset = []
    for claim_id in claim_ids:
        if claim_id not in data:
            print(f"Claim {claim_id} not found in {id}")
            continue
        datapoint = ast.literal_eval(data[claim_id])
        # Get claim, evidence and label as well
        claim_data = convert_from_byte(r.hgetall(claim_id))
        dataset.append({
            'claim_id': claim_id,
            'claim': claim_data['claim'],
            'evidence': claim_data['evidence'],
            'label': claim_data['label'],
            'reasoning': datapoint[0],
            'explanation': datapoint[1]
        })

    with open(f'{id}_{workpackage_id}.json', 'w') as f:
        json.dump(dataset, f)
    
    return dataset

In [23]:
dataset = save_data('guneet', 'workpackage2')

In [27]:
df = pd.DataFrame(dataset)
df['reasoning'].value_counts()

reasoning
deductive    89
abductive    11
Name: count, dtype: int64

In [28]:
with open('chris_workpackage2.json') as f:
    chris_workpackage2 = json.load(f)

with open('sara_workpackage2.json') as f:
    sara_workpackage2 = json.load(f)

with open('vishal_workpackage2.json') as f:
    vishal_workpackage2 = json.load(f)

# convert to dataframe
df_chris = pd.DataFrame(chris_workpackage2)
df_sara = pd.DataFrame(sara_workpackage2)
df_vishal = pd.DataFrame(vishal_workpackage2)

# compare if claim_id are the same in all three datasets
df_chris['claim_id'].equals(df_sara['claim_id']) and df_sara['claim_id'].equals(df_vishal['claim_id'])


True

In [30]:
chris_abdudctive = np.where(df_chris['reasoning'] == 'abductive')
sara_abdudctive = np.where(df_sara['reasoning'] == 'abductive')
vishal_abdudctive = np.where(df_vishal['reasoning'] == 'abductive')

# check overlapping values
reduce(np.intersect1d, (chris_abdudctive, sara_abdudctive, vishal_abdudctive))

array([9])

In [70]:
# load bleiz and nelly workpackage 2
with open('bleiz_workpackage2.json') as f:
    bleiz_workpackage2 = json.load(f)

with open('nelly_workpackage2.json') as f:
    nelly_workpackage2 = json.load(f)

with open('yazhou_workpackage2.json') as f:
    yazhou_workpackage2 = json.load(f)

# convert to dataframe
df_bleiz = pd.DataFrame(bleiz_workpackage2)
df_nelly = pd.DataFrame(nelly_workpackage2)
df_yazhou = pd.DataFrame(yazhou_workpackage2)

# compare if claim_id are the same in all three datasets 
df_bleiz['claim_id'].equals(df_nelly['claim_id']) and df_bleiz['claim_id'].equals(df_yazhou['claim_id'])

True

In [71]:
# check overlapping values between to arrays
bleiz_abductive = np.where(df_bleiz['reasoning'] == 'abductive')
nelly_abductive = np.where(df_nelly['reasoning'] == 'abductive')
yazhou_abductive = np.where(df_yazhou['reasoning'] == 'abductive')
reduce(np.intersect1d, (nelly_abductive, bleiz_abductive, yazhou_abductive))


array([ 6, 62])

In [68]:
# load anel, yuli and jorge workpackage 2
with open('anel_workpackage2.json') as f:
    anel_workpackage2 = json.load(f)

with open('yuli_workpackage2.json') as f:
    yuli_workpackage2 = json.load(f)

with open('jorge_workpackage2.json') as f:
    jorge_workpackage2 = json.load(f)

# convert to dataframe
df_anel = pd.DataFrame(anel_workpackage2)
df_yuli = pd.DataFrame(yuli_workpackage2)
df_jorge = pd.DataFrame(jorge_workpackage2)

# compare if claim_id are the same in all three datasets
df_anel['claim_id'].equals(df_yuli['claim_id']) and df_anel['claim_id'].equals(df_jorge['claim_id'])

True

In [ ]:
# check overlapping values between to arrays
anel_abductive = np.where(df_anel['reasoning'] == 'abductive')
yuli_abductive = np.where(df_yuli['reasoning'] == 'abductive')
jorge_abductive = np.where(df_jorge['reasoning'] == 'abductive')



reduce(np.intersect1d, (anel_abductive, yuli_abductive, jorge_abductive))

array([ 2,  7, 17, 19, 25, 37, 40, 42, 43, 91])

In [34]:
np.shape(jorge_abductive)

(1, 41)

In [40]:
data = (df_anel['reasoning'], df_yuli['reasoning'], df_jorge['reasoning'])
agreements = reduce(np.intersect1d, data)
print(agreements)

['abductive' 'deductive']


In [52]:
agreements = reduce(np.intersect1d, (df_anel['reasoning'], df_yuli['reasoning'], df_jorge['reasoning']))
print(len(agreements))

2


In [55]:
np.shape(np.where((df_anel['reasoning'] == df_yuli['reasoning']) & (df_yuli['reasoning'] == df_jorge['reasoning']))[0])[0]

55

In [ ]:
df_anel['reasoning'] == 'abductive'

0     False
1     False
2      True
3     False
4      True
      ...  
95    False
96    False
97    False
98     True
99    False
Name: reasoning, Length: 100, dtype: bool

array([False,  True])

In [89]:
def bennetts_s(data):
    """
    Calculate Bennett's S for inter-rater reliability.

    Parameters:
        agreements (int): Number of times raters agreed.
        total_ratings (int): Total number of ratings.
        num_categories (int): Number of categories.

    Returns:
        float: Bennett's S value.
    """
    num_categories = len(reduce(np.intersect1d, data))

    total_ratings = len(data[0])

    abductive_data = [np.where(rater == 'abductive')[0] for rater in data]
    deductive_data = [np.where(rater == 'deductive')[0] for rater in data]

    abductive_overlap = reduce(np.intersect1d, abductive_data)
    deductive_overlap = reduce(np.intersect1d, deductive_data)

    agreements = len(abductive_overlap) + len(deductive_overlap)

    print(
        f"Total ratings: {total_ratings}, agreements: {agreements}, num_categories: {num_categories}"
    )

    # Observed agreement (p_o)
    p_o = agreements / total_ratings
    
    # Expected agreement by chance (p_e)
    p_e = 1 / num_categories
    
    # Bennett's S formula
    s = (p_o - p_e) / (1 - p_e)
    return s


In [100]:
# Phemeplus
bennetts_s((df_bleiz['reasoning'], df_nelly['reasoning'], df_yazhou['reasoning']))

Total ratings: 100, agreements: 75, num_categories: 2


0.5

In [97]:
# VITC
bennetts_s((df_sara['reasoning'], df_chris['reasoning'], df_vishal['reasoning']))

Total ratings: 100, agreements: 81, num_categories: 2


0.6200000000000001

In [101]:
# Climate Fever
bennetts_s((df_yuli['reasoning'], df_jorge['reasoning'], df_anel['reasoning']))

Total ratings: 100, agreements: 55, num_categories: 2


0.10000000000000009